# Ab Test Sample Size Instax

A/B-тест для непрерывной метрики на исторических продажах Instax.

Что делает файл:
1) загружает исторические данные;
2) считает baseline, MDE и размер выборки;
3) через Monte Carlo проверяет, что выбранный sample size
   действительно позволяет детектить эффект.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
from scipy.stats import genhyperbolic, mannwhitneyu

## 1. Загрузка исторических данных

Исторические продажи.

index_col=False: не используем ни один столбец как индекс при чтении.

In [ ]:
sales_df = pd.read_csv('data/instax_sales_transactions.csv', sep=',', header=0, index_col=False)

Берем целевую непрерывную метрику.

In [ ]:
sales_series = sales_df['Total_Penjualan']

## 2. Проверка идеи о нормальности средних

In [ ]:
# Этот блок нужен, чтобы визуально проверить центральную предельную теорему:
# если многократно брать выборки и смотреть на их средние,
# распределение средних может приблизиться к нормальному.
sample_means = []

for _ in range(1000):
    sample_means.append(sales_series.sample(1000, replace=True).mean())

sample_means = pd.Series(sample_means)
stats.probplot(sample_means, dist='norm', plot=plt)
plt.title('QQ-plot для распределения средних')
plt.show()

## 3. Расчет baseline, MDE и размера выборки

baseline — среднее историческое значение метрики.

In [ ]:
baseline = sales_series.mean()

MDE (minimum detectable effect) — минимальный эффект, который хотим надежно поймать.

In [ ]:
mde = 50000

Для TTestIndPower effect size задается как MDE / стандартное отклонение.

In [ ]:
effect_size = mde / sales_series.std()

solve_power подбирает sample size на группу при:

power=0.80  -> мощность теста 80%

alpha=0.05  -> уровень значимости 5%

ratio=1     -> группы A и B одинакового размера

In [ ]:
sample_size = round(
    sms.TTestIndPower().solve_power(effect_size, power=0.80, alpha=0.05, ratio=1)
)

In [ ]:
print(baseline)
print(sample_size)

## 4. Monte Carlo-проверка подобранного sample size

По результатам подбора формы распределения было решено использовать genhyperbolic.
Ниже создаются:
- data_a: распределение "как в истории";
- data_b: то же распределение, но со сдвигом среднего на величину MDE.

In [ ]:
data_a = genhyperbolic(p=-0.76, a=17.92, b=17.91, loc=16336.52, scale=16753.85)
data_b = genhyperbolic(p=-0.76, a=17.92, b=17.91, loc=16336.52 + mde, scale=16753.85)

Сценарий 1.

Есть реальная разница между группами -> хотим, чтобы p-value < 0.05 возникало часто.

In [ ]:
pvalues_with_effect = []

In [ ]:
for _ in range(1000):
    group_a = data_a.rvs(size=sample_size)
    group_b = data_b.rvs(size=sample_size)
    _, pvalue = mannwhitneyu(group_a, group_b, alternative='two-sided')
    pvalues_with_effect.append(pvalue)

In [ ]:
pvalues_with_effect = pd.Series(pvalues_with_effect)
print((pvalues_with_effect < 0.05).mean())

Сценарий 2.

Разницы между группами нет -> хотим, чтобы p-value < 0.05 возникало редко.

In [ ]:
pvalues_without_effect = []

In [ ]:
for _ in range(1000):
    group_a = data_a.rvs(size=sample_size)
    group_b = data_a.rvs(size=sample_size)
    _, pvalue = mannwhitneyu(group_a, group_b, alternative='two-sided')
    pvalues_without_effect.append(pvalue)

In [ ]:
pvalues_without_effect = pd.Series(pvalues_without_effect)
print((pvalues_without_effect < 0.05).mean())

## 5. Интерпретация результата

Идея вывода:

если в первом сценарии доля p-value < 0.05 высокая,
а во втором — близка к 5%, то sample size подобран разумно.

В исходной работе вывод был такой:

- нужный MDE достигается даже с запасом;

- точность зависит от того, насколько хорошо распределение смоделировано;

- по форме genhyperbolic оказался наиболее похож на исторические данные.

## 6. Заготовка для подбора формы распределения

Ниже оставлен исходный блок для поиска распределения,
которое лучше всего аппроксимирует исторические данные.

Он закомментирован, потому что нужен не на каждом запуске.

In [ ]:
import warnings
import numpy as np
import scipy.stats as st
from scipy.stats._continuous_distns import _distn_names
import matplotlib

matplotlib.rcParams['figure.figsize'] = (16.0, 12.0)
matplotlib.style.use('ggplot')


def best_fit_distribution(data, bins=200, ax=None):
    # Считаем гистограмму исторических данных и пытаемся подобрать
    # распределение с минимальной ошибкой аппроксимации.
    y, x = np.histogram(data, bins=bins, density=True)
    x = (x + np.roll(x, -1))[:-1] / 2.0

    best_distributions = []

    for i, distribution_name in enumerate(
        [name for name in _distn_names if name not in ['levy_stable', 'studentized_range']]
    ):
        print(f"{i + 1:>3} / {len(_distn_names):<3}: {distribution_name}")
        distribution = getattr(st, distribution_name)

        try:
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                params = distribution.fit(data)

                arg = params[:-2]
                loc = params[-2]
                scale = params[-1]

                pdf = distribution.pdf(x, loc=loc, scale=scale, *arg)
                sse = np.sum(np.power(y - pdf, 2.0))

                try:
                    if ax is not None:
                        pd.Series(pdf, x).plot(ax=ax)
                except Exception:
                    pass

                best_distributions.append((distribution, params, sse))
        except Exception:
            pass

    return sorted(best_distributions, key=lambda item: item[2])


def make_pdf(distribution, params, size=10000):
    arg = params[:-2]
    loc = params[-2]
    scale = params[-1]

    start = distribution.ppf(0.01, *arg, loc=loc, scale=scale) if arg else distribution.ppf(0.01, loc=loc, scale=scale)
    end = distribution.ppf(0.99, *arg, loc=loc, scale=scale) if arg else distribution.ppf(0.99, loc=loc, scale=scale)

    x = np.linspace(start, end, size)
    y = distribution.pdf(x, loc=loc, scale=scale, *arg)
    return pd.Series(y, x)


data = sales_series

plt.figure(figsize=(12, 8))
ax = data.plot(
    kind='hist',
    bins=50,
    density=True,
    alpha=0.5,
    color=list(matplotlib.rcParams['axes.prop_cycle'])[1]['color']
)

data_ylim = ax.get_ylim()

best_distributions = best_fit_distribution(data, 200, ax)
best_distribution = best_distributions[0]

ax.set_ylim(data_ylim)
ax.set_title('All fitted distributions')
ax.set_xlabel('Metric value')
ax.set_ylabel('Frequency')
plt.show()

pdf = make_pdf(best_distribution[0], best_distribution[1])

plt.figure(figsize=(12, 8))
ax = pdf.plot(lw=2, label='PDF', legend=True)
data.plot(kind='hist', bins=50, density=True, alpha=0.5, label='Data', legend=True, ax=ax)

param_names = (
    (best_distribution[0].shapes + ', loc, scale').split(', ')
    if best_distribution[0].shapes else ['loc', 'scale']
)
param_str = ', '.join([f'{key}={value:0.2f}' for key, value in zip(param_names, best_distribution[1])])
dist_str = f'{best_distribution[0].name}({param_str})'

ax.set_title('Best fit distribution\n' + dist_str)
ax.set_xlabel('Metric value')
ax.set_ylabel('Frequency')
plt.show()

fit_table = pd.DataFrame(best_distributions).sort_values(2)
print(fit_table[0][0], fit_table[1][0])

---
**Примечание:** для запуска ноутбука достаточно открыть его в корне соответствующего репозитория — все файлы данных лежат рядом и читаются по относительному пути.